# 01 - GCP account, project, and budget

This notebook creates the first auditable foundation for the Chess Lakehouse project. It runs the Google Cloud CLI from your local machine and records the commands and verification output.

## Outcome

1. Confirm the local Google Cloud CLI installation.
2. Sign in interactively and verify the selected account.
3. Create a uniquely named GCP project.
4. Link the project to an existing free-trial Cloud Billing account.
5. Create a monthly project-scoped budget with email thresholds.
6. Verify every resulting resource.

> Important: a Cloud Billing budget is an alert, not a hard spending cap. Authentication and initial free-trial/billing-account enrollment still require Google's interactive browser flow. Do not commit executed notebook output if it exposes your email address, billing account ID, or project metadata.


## 0. Prerequisites

Install the Google Cloud CLI and Jupyter locally before continuing. On Windows, reopen your terminal after installing the CLI so `gcloud` is available on `PATH`. Then run this notebook using the same Windows user account.

Create or activate the Google Cloud free trial in the browser before the billing-link step. The notebook can list and link an existing billing account; it does not create the payments profile or accept free-trial terms for you.


In [ ]:
from __future__ import annotations

import json
import re
import shutil
import subprocess
from datetime import datetime


def run_gcloud(*args: str, capture: bool = False) -> str:
    executable = shutil.which("gcloud") or shutil.which("gcloud.cmd")
    if not executable:
        raise RuntimeError("gcloud was not found on PATH. Install Google Cloud CLI and restart Jupyter.")
    command = [executable, *args]
    print("$ gcloud", " ".join(args))
    result = subprocess.run(command, check=True, text=True, capture_output=capture)
    if capture and result.stdout:
        print(result.stdout.rstrip())
    return result.stdout.strip() if capture else ""


run_gcloud("version")


## 1. Set project parameters

Choose a globally unique project ID. It must be 6-30 characters, begin with a lowercase letter, and contain only lowercase letters, digits, and hyphens. Project IDs cannot be changed after creation.

Keep `RUN_MUTATIONS = False` for the first pass. Review the generated values and read-only checks, then change it to `True` only when you intend to create cloud resources.


In [ ]:
PROJECT_ID = "chess-lakehouse-CHANGE-ME"  # lowercase and globally unique
PROJECT_NAME = "Chess Analytics Lakehouse"
BUDGET_AMOUNT = "1000INR"  # choose your own monthly alert amount
BUDGET_NAME = "Chess Lakehouse Monthly Budget"
RUN_MUTATIONS = False

if PROJECT_ID == "chess-lakehouse-CHANGE-ME":
    raise ValueError("Replace PROJECT_ID with a globally unique lowercase ID before continuing.")
if not re.fullmatch(r"[a-z][a-z0-9-]{5,29}", PROJECT_ID):
    raise ValueError("PROJECT_ID does not satisfy the basic GCP format rules.")
if not re.fullmatch(r"[1-9][0-9]*(?:\.[0-9]{1,2})?[A-Z]{3}", BUDGET_AMOUNT):
    raise ValueError("Use a budget such as 1000INR or 25USD.")

print({
    "project_id": PROJECT_ID,
    "project_name": PROJECT_NAME,
    "budget_amount": BUDGET_AMOUNT,
    "mutations_enabled": RUN_MUTATIONS,
})


## 2. Authenticate

Run the login command once. It opens Google's browser-based authorization flow. Never paste access tokens, passwords, service-account keys, or payment information into this notebook.


In [ ]:
# Uncomment only when you need to sign in or change accounts.
# run_gcloud("auth", "login")

run_gcloud("auth", "list", "--filter=status:ACTIVE", "--format=table(account,status)")


## 3. Create and select the project

This is the first mutating operation. If creation reports that the ID is already in use, choose a different `PROJECT_ID`; do not add personal data such as your birth date or phone number.


In [ ]:
if not RUN_MUTATIONS:
    raise RuntimeError("Guard active. Review the notebook, then set RUN_MUTATIONS = True to create the project.")

run_gcloud("projects", "create", PROJECT_ID, "--name", PROJECT_NAME)
run_gcloud("config", "set", "project", PROJECT_ID)
run_gcloud("projects", "describe", PROJECT_ID, "--format=json", capture=True)


## 4. Select and link the Cloud Billing account

The first cell lists billing accounts visible to the signed-in user. Copy the intended **open** account ID into `BILLING_ACCOUNT_ID`. Treat that ID as private operational metadata and clear cell outputs before publishing.


In [ ]:
run_gcloud("billing", "accounts", "list", "--filter=open=true", "--format=table(name,displayName,open)")


In [ ]:
BILLING_ACCOUNT_ID = "XXXXXX-XXXXXX-XXXXXX"  # copy only the ID, without 'billingAccounts/'

if not re.fullmatch(r"[0-9A-F]{6}-[0-9A-F]{6}-[0-9A-F]{6}", BILLING_ACCOUNT_ID):
    raise ValueError("Replace BILLING_ACCOUNT_ID with an open account returned by the previous cell.")
if not RUN_MUTATIONS:
    raise RuntimeError("Guard active. Set RUN_MUTATIONS = True only when you intend to link billing.")

run_gcloud("billing", "projects", "link", PROJECT_ID, "--billing-account", BILLING_ACCOUNT_ID)
run_gcloud("billing", "projects", "describe", PROJECT_ID, "--format=json", capture=True)


## 5. Create a project-scoped monthly budget

The budget uses actual-spend email thresholds at 50%, 80%, 90%, and 100%. Default IAM recipients remain enabled, so eligible billing administrators and users receive alerts. Pub/Sub automation is intentionally deferred to avoid extra moving parts and possible charges in milestone 1.


In [ ]:
if not RUN_MUTATIONS:
    raise RuntimeError("Guard active. Set RUN_MUTATIONS = True only when you intend to enable the API and create the budget.")

run_gcloud("services", "enable", "billingbudgets.googleapis.com", "--project", PROJECT_ID)
project_number = run_gcloud(
    "projects", "describe", PROJECT_ID, "--format=value(projectNumber)", capture=True
).splitlines()[-1]

run_gcloud(
    "billing", "budgets", "create",
    "--billing-account", BILLING_ACCOUNT_ID,
    "--display-name", BUDGET_NAME,
    "--budget-amount", BUDGET_AMOUNT,
    "--calendar-period", "month",
    "--filter-projects", f"projects/{project_number}",
    "--threshold-rule", "percent=0.50,basis=current-spend",
    "--threshold-rule", "percent=0.80,basis=current-spend",
    "--threshold-rule", "percent=0.90,basis=current-spend",
    "--threshold-rule", "percent=1.00,basis=current-spend",
)


## 6. Final verification

All checks below are read-only. Confirm the project is ACTIVE, billing is enabled, and exactly one intended budget is shown. A newly created budget or alert can take time to appear or send its first notification.


In [ ]:
run_gcloud("projects", "describe", PROJECT_ID, "--format=table(projectId,name,projectNumber,lifecycleState)")
run_gcloud("billing", "projects", "describe", PROJECT_ID, "--format=table(projectId,billingAccountName,billingEnabled)")
run_gcloud("billing", "budgets", "list", "--billing-account", BILLING_ACCOUNT_ID, "--format=table(displayName,amount.specifiedAmount.currencyCode,amount.specifiedAmount.units)")


## 7. GitHub evidence checklist

Before committing this notebook:

- Clear all cell outputs (`Kernel` -> `Restart Kernel and Clear All Outputs`).
- Restore `RUN_MUTATIONS = False`.
- Replace the real billing account ID with the placeholder.
- Keep the chosen public-safe project ID if you want it documented.
- Capture separate screenshots of the project overview and budget page only after hiding account/payment details.
- Record the date, selected region policy (later milestone), budget amount, and verification result in the repository README.

Milestone 1 is complete only when the project exists, billing is linked, the budget is visible, and the read-only verification cells succeed.
